# ROBERTA FINE-TUNING

In [14]:
from collections import Counter
import json

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []

    for key, value in data.items():
        tweet_text = value['tweet']
        votos = value['labels_task1']
        
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue 
            
        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])

    return X, y, ids



ruta = 'dataset_task1_exist2025/training.json' 
try:
    X_train, y_train, ids_train = cargar_datos_entrenamiento(ruta)
    
    print(f"Procesados {len(X_train)} tweets correctamente.")
    print(f"Ejemplo - Texto: {X_train[0][:50]}...")
    print(f"Ejemplo - Label: {y_train[0]}")

except FileNotFoundError:
    print("Error: No se encontró el archivo training.json. Revisa la ruta.")

Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import evaluate

model_id = "xlm-roberta-base" 

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Formato Hugging Face
label_map = {"YES": 1, "NO": 0}
train_dict = {
    "text": X_train, 
    "label": [label_map[l] for l in y_train]
}
ds = Dataset.from_dict(train_dict)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=128)

tokenized_ds = ds.map(preprocess_function, batched=True)

Map: 100%|██████████| 6064/6064 [00:00<00:00, 12673.91 examples/s]


In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold
import numpy as np

metric = evaluate.load("f1")
# Split 80/20 antes de entrenar
split_ds = tokenized_ds.train_test_split(test_size=0.2, seed=42)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8,
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

def model_init(trial=None):
    base_model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    return get_peft_model(base_model, peft_config)

# Espacio de búsqueda con Optuna
def my_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
    }


trainer_hpo = Trainer(
    model_init=model_init,
    args=TrainingArguments(
        output_dir="./hpo_results",
        eval_strategy="epoch",
        save_strategy="no",
        disable_tqdm=False, 
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=10,
        fp16=True if torch.cuda.is_available() else False
    ),
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Iniciando búsqueda de hiperparámetros...")
best_run = trainer_hpo.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=my_hp_space,
    n_trials=5 
)

# Validación cruzada
print(f"\nMejores parámetros: {best_run.hyperparameters}")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

labels = np.array(tokenized_ds["label"])

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n---VALIDANDO FOLD {fold + 1} ---")
    
    final_args = TrainingArguments(
        output_dir=f"./fold_{fold}",
        learning_rate=best_run.hyperparameters["learning_rate"],
        num_train_epochs=best_run.hyperparameters["num_train_epochs"],
        per_device_train_batch_size=best_run.hyperparameters["per_device_train_batch_size"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        warmup_ratio=0.1,
        logging_steps=20,
        fp16=True if torch.cuda.is_available() else False
    )

    trainer_cv = Trainer(
        model_init=model_init,
        args=final_args,
        train_dataset=tokenized_ds.select(train_idx),
        eval_dataset=tokenized_ds.select(val_idx),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    
    trainer_cv.train()
    eval_res = trainer_cv.evaluate()
    cv_scores.append(eval_res["eval_f1"])
    print(f"Fold {fold+1} F1: {eval_res['eval_f1']:.4f}")

print(f"\nRESULTADO FINAL CV: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1257.28it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- M

Iniciando búsqueda de hiperparámetros...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1238.23it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Epoch,Training Loss,Validation Loss,F1
1,0.487067,0.426265,0.803610
2,0.452518,0.402363,0.817902
3,0.372585,0.400752,0.823697


[I 2026-03-04 10:48:50,442] Trial 0 finished with value: 0.8236966954614013 and parameters: {'learning_rate': 0.000448218550566169, 'num_train_epochs': 3, 'per_device_train_batch_size': 32}. Best is trial 0 with value: 0.8236966954614013.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1299.70it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.ou

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Fold 4 F1: 0.8385

---VALIDANDO FOLD 5 ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1261.22it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Epoch,Training Loss,Validation Loss,F1
1,0.477417,0.476271,0.773280
2,0.416234,0.553552,0.742018
3,0.419454,0.410426,0.822030


Fold 5 F1: 0.8220

RESULTADO FINAL CV: 0.8198 (+/- 0.0114)


In [ ]:
import json
import torch
from transformers import DataCollatorWithPadding

print("Reentrenando modelo final con el dataset completo...")

final_training_args = TrainingArguments(
    output_dir="./modelo_final_exist",
    learning_rate=best_run.hyperparameters["learning_rate"],
    num_train_epochs=best_run.hyperparameters["num_train_epochs"],
    per_device_train_batch_size=best_run.hyperparameters["per_device_train_batch_size"],
    warmup_ratio=0.1,
    fp16=True if torch.cuda.is_available() else False,
    save_strategy="no" 
)

trainer_final = Trainer(
    model_init=model_init,
    args=final_training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

trainer_final.train()

ruta_test = 'dataset_task1_exist2025/test.json'
with open(ruta_test, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

ids_test = []
textos_test = []
for key, value in test_data.items():
    ids_test.append(value['id_EXIST'])
    textos_test.append(value['tweet'])

print("Generando predicciones para el test oficial...")
model_final = trainer_final.model
model_final.eval()
model_final.to("cuda" if torch.cuda.is_available() else "cpu")

predicciones_raw = []

batch_size = 16
for i in range(0, len(textos_test), batch_size):
    batch_texts = textos_test[i:i+batch_size]
    inputs = tokenizer(batch_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(model_final.device)
    
    with torch.no_grad():
        outputs = model_final(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1)
        predicciones_raw.extend(preds.cpu().tolist())

label_map_inv = {1: "YES", 0: "NO"}
output_json = []

for id_ev, pred_idx in zip(ids_test, predicciones_raw):
    output_json.append({
        'test_case': 'EXIST2025',
        'id': id_ev,
        'value': label_map_inv[pred_idx]
    })

# Guardar archivo
nombre_archivo = 'Thinkpol_modelo_ROBERTA_LoRA.json'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=4)

print(f"Proceso finalizado. Archivo {nombre_archivo} listo para entrega.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Reentrenando modelo final con el dataset completo...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1506.60it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Step,Training Loss
500,0.489377


Generando predicciones para el test oficial...
Proceso finalizado. Archivo Thinkpol_modelo_ROBERTA_LoRA.json listo para entrega.
